In [52]:
import os

os.makedirs("app", exist_ok=True)
os.makedirs("tests", exist_ok=True)

print("Folders created")

Folders created


In [53]:
%%writefile requirements.txt
fastapi
uvicorn
pandas
numpy
scikit-learn
joblib
pytest
httpx

Overwriting requirements.txt


In [54]:
import joblib

model = joblib.load("fresh_model.pkl")

print(type(model))
print(model.n_features_in_)
print(model.feature_names_in_)

<class 'sklearn.linear_model._logistic.LogisticRegression'>
13
['gross_amount' 'quantity' 'returned' 'rating' 'discount_pct'
 'delivery_days' 'city_tier' 'age_group' 'acquisition_channel'
 'loyalty_tier' 'preferred_category' 'skin_type' 'marketing_consent']


In [55]:
%%writefile app/main.py

from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

# Load model
model = joblib.load("fresh_model.pkl")

app = FastAPI(title="Churn Prediction API")


class CustomerData(BaseModel):
    gross_amount: float
    quantity: int
    returned: float
    rating: float
    discount_pct: float
    delivery_days: float
    city_tier: int
    age_group: int
    acquisition_channel: int
    loyalty_tier: int
    preferred_category: int
    skin_type: int
    marketing_consent: int


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict")
def predict(data: CustomerData):

    df = pd.DataFrame([data.dict()])

    prediction = int(model.predict(df)[0])

    probability = float(model.predict_proba(df)[0][1])

    risk = "High Risk" if probability > 0.5 else "Low Risk"

    return {
        "churn_prediction": prediction,
        "churn_probability": round(probability, 4),
        "risk_level": risk
    }


@app.post("/batch_predict")
def batch_predict(data: list[CustomerData]):

    df = pd.DataFrame([item.dict() for item in data])

    predictions = model.predict(df)

    probabilities = model.predict_proba(df)

    results = []

    for pred, prob in zip(predictions, probabilities):

        results.append({
            "prediction": int(pred),
            "probability": round(float(prob[1]), 4)
        })

    return results

Overwriting app/main.py


In [56]:
!cat app/main.py


from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

# Load model
model = joblib.load("fresh_model.pkl")

app = FastAPI(title="Churn Prediction API")


class CustomerData(BaseModel):
    gross_amount: float
    quantity: int
    returned: float
    rating: float
    discount_pct: float
    delivery_days: float
    city_tier: int
    age_group: int
    acquisition_channel: int
    loyalty_tier: int
    preferred_category: int
    skin_type: int
    marketing_consent: int


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict")
def predict(data: CustomerData):

    df = pd.DataFrame([data.dict()])

    prediction = int(model.predict(df)[0])

    probability = float(model.predict_proba(df)[0][1])

    risk = "High Risk" if probability > 0.5 else "Low Risk"

    return {
        "churn_prediction": prediction,
        "churn_probability": round(probability, 4),
        "risk_level": risk
    }


@app.post("/batch

In [57]:
from app.main import CustomerData

sample = {
    "gross_amount": 1000.0,
    "quantity": 2,
    "returned": 0.0,
    "rating": 4.5,
    "discount_pct": 0.2,
    "delivery_days": 3,

    "city_tier":0,
    "age_group":1,
    "acquisition_channel":2,
    "loyalty_tier":3,
    "preferred_category":3,
    "skin_type":0,
    "marketing_consent":1
}

In [58]:
CustomerData(**sample)

CustomerData(gross_amount=1000.0, quantity=2, returned=0.0, rating=4.5, discount_pct=0.2, delivery_days=3.0, city_tier=0, age_group=1, acquisition_channel=2, loyalty_tier=3, preferred_category=3, skin_type=0, marketing_consent=1)

In [59]:
!cat app/main.py


from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

# Load model
model = joblib.load("fresh_model.pkl")

app = FastAPI(title="Churn Prediction API")


class CustomerData(BaseModel):
    gross_amount: float
    quantity: int
    returned: float
    rating: float
    discount_pct: float
    delivery_days: float
    city_tier: int
    age_group: int
    acquisition_channel: int
    loyalty_tier: int
    preferred_category: int
    skin_type: int
    marketing_consent: int


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/predict")
def predict(data: CustomerData):

    df = pd.DataFrame([data.dict()])

    prediction = int(model.predict(df)[0])

    probability = float(model.predict_proba(df)[0][1])

    risk = "High Risk" if probability > 0.5 else "Low Risk"

    return {
        "churn_prediction": prediction,
        "churn_probability": round(probability, 4),
        "risk_level": risk
    }


@app.post("/batch

In [60]:
import importlib
import app.main
importlib.reload(app.main)
from app.main import CustomerData

In [61]:
sample = {
    "gross_amount":1000.0,
    "quantity":2,
    "returned":0.0,
    "rating":4.5,
    "discount_pct":0.2,
    "delivery_days":3,
    "city_tier":0,
    "age_group":1,
    "acquisition_channel":2,
    "loyalty_tier":3,
    "preferred_category":3,
    "skin_type":0,
    "marketing_consent":1
}

CustomerData(**sample)

CustomerData(gross_amount=1000.0, quantity=2, returned=0.0, rating=4.5, discount_pct=0.2, delivery_days=3.0, city_tier=0, age_group=1, acquisition_channel=2, loyalty_tier=3, preferred_category=3, skin_type=0, marketing_consent=1)

In [62]:
from app.main import predict

result = predict(CustomerData(**sample))

print(result)

{'churn_prediction': 1, 'churn_probability': 0.6546, 'risk_level': 'High Risk'}


In [63]:
%%writefile tests/test_api.py

from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

sample = {
    "gross_amount":1000.0,
    "quantity":2,
    "returned":0.0,
    "rating":4.5,
    "discount_pct":0.2,
    "delivery_days":3,
    "city_tier":0,
    "age_group":1,
    "acquisition_channel":2,
    "loyalty_tier":3,
    "preferred_category":3,
    "skin_type":0,
    "marketing_consent":1
}

def test_health():
    response = client.get("/health")
    assert response.status_code == 200

def test_predict():
    response = client.post("/predict", json=sample)
    assert response.status_code == 200

def test_batch_predict():
    response = client.post("/batch_predict", json=[sample])
    assert response.status_code == 200

Overwriting tests/test_api.py


In [64]:
!pytest tests/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
configfile: pytest.ini
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.9
collected 3 items                                                              

tests/test_api.py::test_health PASSED                                    [ 33%]
tests/test_api.py::test_predict PASSED                                   [ 66%]
tests/test_api.py::test_batch_predict PASSED                             [100%]

============================== 3 passed in 1.59s ===============================


In [65]:
%%writefile README.md

# FastAPI Churn Scoring Service

## Overview
This project provides a FastAPI service for predicting customer churn.
Model: Logistic Regression

## Endpoints

### GET /health
Returns API health status.

### POST /predict
Predicts churn for a single customer.

### POST /batch_predict
Predicts churn for multiple customers.

## Run

uvicorn app.main:app --reload

## Run Tests

pytest tests/test_api.py -v

## Sample Request

{
    "gross_amount":1000.0,
    "quantity":2,
    "returned":0.0,
    "rating":4.5,
    "discount_pct":0.2,
    "delivery_days":3,
    "city_tier":0,
    "age_group":1,
    "acquisition_channel":2,
    "loyalty_tier":3,
    "preferred_category":3,
    "skin_type":0,
    "marketing_consent":1
}

## Sample Response

{
    "churn_prediction":1,
    "churn_probability":0.6546,
    "risk_level":"High Risk"
}

Overwriting README.md


In [66]:
%%writefile monitoring_plan.md

# Monitoring Plan

## Metrics to Track

1. Prediction distribution
   - Percentage of churn predictions
   - Percentage of non-churn predictions

2. Data drift
   - Monitor changes in customer feature distributions

3. API performance
   - Response time
   - Failed requests
   - Error rates

4. Business outcomes
   - Retention campaign success
   - Actual customer churn rate

## Retraining Triggers

- Significant drop in model performance
- Data drift detected
- Quarterly model review

# Responsible Use

The model should support retention decisions.

The model should not be used as the sole reason for customer targeting or exclusion.

Human review should always be considered before business action.

Overwriting monitoring_plan.md


In [67]:
!ls

app		 model.pkl	     README.md	       tests
fresh_model.pkl  monitoring_plan.md  requirements.txt
model_new.pkl	 pytest.ini	     sample_data


# Conclusion
In this part of the project, I developed a FastAPI-based churn prediction service using the Logistic Regression model created earlier. The API includes endpoints to check service health, predict churn for a single customer, and generate predictions for multiple customers at once. Input validation was handled using Pydantic to ensure that only valid data is accepted. I also created and ran API tests to confirm that all endpoints work correctly. Finally, I included a requirements file for reproducibility, along with a monitoring and responsible-use plan to support future deployment and maintenance of the service.